# Notebook 01 — Data Quality Assessment & Cleaning Pipeline
## NovaCred Credit Application Governance Audit

**Dataset:** `raw_credit_applications.json` — 502 raw records in nested JSON format  
**Objective:** Identify, quantify, and remediate all data quality issues before the data passes to the Data Scientist and Governance Officer.

---
### Data Engineer To-Do List

| # | Task | Data Quality Dimension | Status |
|---|------|----------------------|--------|
| 1 | Load raw JSON and flatten nested structure | — | ⬜ |
| 2 | Detect and remove duplicate records | Completeness / Uniqueness | ⬜ |
| 3 | Standardise inconsistent gender coding | Consistency | ⬜ |
| 4 | Fix string-typed income values → numeric | Validity / Data Type | ⬜ |
| 5 | Handle missing income (null) values | Completeness | ⬜ |
| 6 | Fix impossible values (negative credit history months) | Validity | ⬜ |
| 7 | Fix out-of-range DTI ratio (> 1.0) | Validity | ⬜ |
| 8 | Normalise 3 inconsistent date formats → ISO 8601 | Consistency | ⬜ |
| 9 | Handle missing date-of-birth values | Completeness | ⬜ |
| 10 | Flag missing PII fields (email, SSN) | Completeness | ⬜ |
| 11 | Produce final cleaned dataset & quality report | — | ⬜ |


### 1. Imports and configuration

In [125]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1: Import all the libraries (tools) we need
# ═══════════════════════════════════════════════════════════════════════════

# json: reads our raw data file which is stored in JSON format
import json

# re: "regular expressions" — a powerful way to check if text matches a pattern, We will use it to detect which date format a string is in (e.g. DD/MM/YYYY)
import re

# warnings: Python sometimes prints yellow warning messages that are not errors but clutter the output. This line lets us silence them.
import warnings

# numpy: fast numerical computing. We use it mainly for np.nan (a special "not a number" value) and np.polyfit (fitting a line through data points)
import numpy as np

# pandas: THE library for working with tables of data in Python. A DF is a basically a excel spreadsheet in memory.
import pandas as pd

# matplotlib & Seaborn: the foundational Python charting library and a popular extension that makes it easier to create nice-looking charts.
import matplotlib.pyplot as plt
import seaborn as sns

# datetime: Python's built-in module for working with dates.
from datetime import datetime

# Counter: a convenient tool that counts how many times each value appears
# in a list. E.g. Counter(['Male','Male','Female','M']) → {'Male':2,'Female':1,'M':1}
from collections import Counter


warnings.filterwarnings('ignore')

# ─── Set the visual style for all charts ────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)


# ─── File paths ─────────────────────────────────────────────────────────────
# Instead of typing the full file path over and over, we store them in variables once. If the file moves, we only change it here.

RAW_PATH    = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/raw_credit_applications.json'  # the original, untouched file
CLEAN_PATH  = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/cleaned_credit_applications.csv'  # where we save our clean output
REPORT_PATH = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/data_quality_report.json'  # our written log of every issue found
FIGURES_DIR = '/Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/figures/'  # folder where charts are saved

# ─── Audit reference date ────────────────────────────────────────────────────
# We need a fixed "today" to calculate how old each applicant is from their
# date of birth. We use March 1, 2026 — the approximate date of this audit.
AUDIT_DATE = datetime(2026, 3, 1)

# A quick confirmation message so we know the cell ran without errors
print(f'✓ All libraries loaded successfully')
print(f'✓ Audit reference date set to: {AUDIT_DATE.strftime("%B %d, %Y")}')
print(f'✓ Raw data will be read from : {RAW_PATH}')
print(f'✓ Clean data will be saved to: {CLEAN_PATH}')


✓ All libraries loaded successfully
✓ Audit reference date set to: March 01, 2026
✓ Raw data will be read from : /Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/raw_credit_applications.json
✓ Clean data will be saved to: /Users/juliuscaspar/Downloads/CV/Universitys/NOVA MASTERS/Data Ecosystems & and Governance/Group Data Gov/Datagov group 2/dego-project-team2/data/cleaned_credit_applications.csv


## Section 2  Loading the Raw Data & Converting It Into a Table


raw data currnetly is in `raw_credit_applications.json`. Here is what one credit application looks like inside the JSON file:

```json
{
  "_id": "app_001",
  "applicant_info": {
    "full_name": "Stephanie Nguyen",
    "email": "s.nguyen@gmail.com",
    "gender": "Female",
    "date_of_birth": "1990-03-15"
  },
  "financials": {
    "annual_income": 85000,
    "credit_history_months": 72,
    "debt_to_income": 0.24
  },
  "decision": {
    "loan_approved": true,
    "interest_rate": 4.2
  }
}

The data is **nested** . The application record contains `applicant_info`, which itself contains `full_name`, `email`, etc. Pandas cannot work directly with this structure, it needs a flat table where every piece of information is its own column.

In [126]:
# open(RAW_PATH, 'r') — opens the file at the path we defined above'r' means "read mode" — we are only reading, not writing to it

# The 'with' keyword ensures the file is automatically closed when we are done reading it, even if an error occurs. This prevents memory leaks.

# json.load(f) — reads the text inside the file and converts it into a Python object (in this case, a list of dictionaries — one per application)

with open(RAW_PATH, 'r') as f:
    raw_data = json.load(f)

print(f'Raw records loaded: {len(raw_data):,}')
print(f'Example record keys: {list(raw_data[0].keys())}')
print(f'applicant_info keys: {list(raw_data[0]["applicant_info"].keys())}')
print(f'financials keys    : {list(raw_data[0]["financials"].keys())}')
print(f'decision keys      : {list(raw_data[0]["decision"].keys())}')
print(f'spending_behavior  : {raw_data[0]["spending_behavior"][:2]}')
# spending_behavior is a LIST of dictionaries, not a single value  That is why it needs special handling when we flatten

Raw records loaded: 502
Example record keys: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']
applicant_info keys: ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
financials keys    : ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
decision keys      : ['loan_approved', 'rejection_reason']
spending_behavior  : [{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790}]


In [127]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3: Define the function that flattens one JSON record into one table row
# ═══════════════════════════════════════════════════════════════════════════

def flatten_record(record: dict) -> dict:
    """
    Takes one nested JSON credit application and returns a flat dictionary.

    Parameters
    ----------
    record : dict
        One raw application record as loaded from JSON.

    Returns
    -------
    dict
        A flat dictionary where every value is a simple scalar (number,
        string, boolean, or None) — ready to become one row in a DataFrame.
    """

    # ── Step A: Extract each nested sub-section into its own variable ────────
    #
    # record.get('applicant_info', {}) means:
    #   "give me the value stored under the key 'applicant_info'"
    #   "if that key doesn't exist, return an empty dictionary {} instead of crashing"

    ai  = record.get('applicant_info', {})   # personal details
    fin = record.get('financials', {})        # financial details
    dec = record.get('decision', {})          # loan decision
    sb  = record.get('spending_behavior', []) # list of spending entries (default: empty list)

    # ── Step B: Summarise the spending_behavior list ─────────────────────────
    #
    # spending_behavior is a LIST like this:
    # [{"category": "Rent", "amount": 800}, {"category": "Alcohol", "amount": 120}]
    #
    # We cannot put a list inside a single table cell, so we compute two scalar
    # summaries from it:
    #   1. total_spend  — sum of ALL spending categories
    #   2. alcohol_spend — sum of ONLY the "Alcohol" category

    # sum() adds up all the values we give it.
    # The expression inside is a "generator" — it loops over every item in sb,
    # and for each item checks:
    #   - Does item have an 'amount'? → item.get('amount', 0) returns 0 if missing
    #   - Is that amount actually a number? → isinstance(..., (int, float)) returns True/False.

    total_spend = sum(
        item.get('amount', 0)
        for item in sb
        if isinstance(item.get('amount'), (int, float))
    )

    # ── Step C: Build and return the flat dictionary ─────────────────────────
    #
    # Each line is:  'column_name' : value_to_put_in_that_column
    #
    # IMPORTANT NAMING CONVENTION:
    # Fields that still need cleaning are called 'field_raw' (e.g., 'gender_raw')
    # This makes it clear which values are original and which are cleaned.
    # We NEVER overwrite the original — we always create a new cleaned column.
    # Reason: if our cleaning logic has a bug, we can always go back to the original.

    return {
        # ── Unique identifier ────────────────────────────────────────────────
        '_id'                    : record.get('_id'),

        # ── Applicant personal info ──────────────────────────────────────────
        'full_name'              : ai.get('full_name'),
        'email'                  : ai.get('email'),
        'ssn'                    : ai.get('ssn'),        # Social Security Number
        'ip_address'             : ai.get('ip_address'), # IP at time of application
        'gender_raw'             : ai.get('gender'),     # RAW — inconsistent (M/Male/F/Female)
        'date_of_birth_raw'      : ai.get('date_of_birth'), # RAW — 3 different formats
        'zip_code'               : ai.get('zip_code'),

        # ── Financial info ───────────────────────────────────────────────────
        'annual_income_raw'      : fin.get('annual_income'),  # RAW — some are strings!
        'credit_history_months'  : fin.get('credit_history_months'), # some are negative!
        'debt_to_income'         : fin.get('debt_to_income'), # one is > 1.0 (impossible)
        'savings_balance'        : fin.get('savings_balance'),

        # ── Loan decision ────────────────────────────────────────────────────
        'loan_approved'          : dec.get('loan_approved'),   # True or False
        'interest_rate'          : dec.get('interest_rate'),   # only present if approved
        'approved_amount'        : dec.get('approved_amount'), # only present if approved
        'rejection_reason'       : dec.get('rejection_reason'),# only present if rejected

        # ── Derived spending features ────────────────────────────────────────
        'total_spending'         : total_spend,
    }

# ─── Apply the function to every record ─────────────────────────────────────
#
# [flatten_record(r) for r in raw_data] is a "list comprehension":
# it loops over every record r in raw_data and applies our function to it.
# The result is a list of 502 flat dictionaries.
#
# pd.DataFrame([...]) takes that list of dictionaries and converts it into
# a DataFrame — a table with rows (applications) and columns (fields).

df_raw = pd.DataFrame([flatten_record(r) for r in raw_data])

# ─── Quick sanity check ───────────────────────────────────────────────────────
print(f'DataFrame created successfully!')
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
# .shape returns (number_of_rows, number_of_columns)

print(f'\nAll column names:')
for col in df_raw.columns:
    print(f'  - {col}')

print(f'\nFirst 2 rows (preview):')
df_raw.head(2)  # .head(2) shows the first 2 rows


DataFrame created successfully!
Shape: 502 rows × 17 columns

All column names:
  - _id
  - full_name
  - email
  - ssn
  - ip_address
  - gender_raw
  - date_of_birth_raw
  - zip_code
  - annual_income_raw
  - credit_history_months
  - debt_to_income
  - savings_balance
  - loan_approved
  - interest_rate
  - approved_amount
  - rejection_reason
  - total_spending

First 2 rows (preview):


,_id,full_name,email,ssn,ip_address,gender_raw,date_of_birth_raw,zip_code,annual_income_raw,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,total_spending
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,NaN,NaN,algorithm_risk_score,1517
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,NaN,NaN,algorithm_risk_score,947


In [128]:
#getting a quick overview of the data types and missing values in each column
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
_id,500,500,app_200,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
full_name,500,475,Susan Flores,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,500,494,,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ssn,496,494,937-72-8731,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_address,496,496,192.168.48.155,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender_raw,500,5,Male,194,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_of_birth_raw,500,494,,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip_code,500,196,10048,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
annual_income_raw,495.0,132.0,79000.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
credit_history_months,500.0,NaN,NaN,NaN,50.496,31.205666,0.0,27.0,48.5,72.0,133.0


## Section 3 — Issue #1: Duplicate Records

In [129]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4: Detect duplicate records
# ═══════════════════════════════════════════════════════════════════════════

# value_counts() counts how many times each unique value appears in a column.
id_counts = df_raw['_id'].value_counts()

# We filter the count series to only show IDs that appear more than once.
dup_ids = id_counts[id_counts > 1].index.tolist()

# Get the actual rows that are duplicated so we can inspect them
dup_records = df_raw[df_raw['_id'].isin(dup_ids)]
# df_raw['_id'].isin(dup_ids) returns True for every row whose _id is in our
# list of duplicate IDs. Wrapping df_raw[...] around it filters the table to
# only those rows.

# Print the summary
print(f'Total records in raw file    : {len(df_raw):,}')
print(f'Number of unique _id values  : {df_raw["_id"].nunique():,}')
# .nunique() = "number of unique values"

print(f'Duplicate _id values found   : {dup_ids}')
print(f'Total rows affected          : {len(dup_records)}')

# Show the duplicate pairs side by side so we can confirm they are identical
print(f'\n--- Inspecting the duplicate pairs ---')
for dup_id in dup_ids:
    # Filter to just the rows with this specific duplicate ID
    subset = df_raw[df_raw['_id'] == dup_id][
        ['_id', 'full_name', 'annual_income_raw', 'loan_approved']
    ]
    print(f'\nDuplicate: {dup_id}')
    print(subset.to_string(index=False))
    # to_string(index=False) prints the table without the row numbers on the left
    # which makes the output cleaner and easier to read


Total records in raw file    : 502
Number of unique _id values  : 500
Duplicate _id values found   : ['app_042', 'app_001']
Total rows affected          : 4

--- Inspecting the duplicate pairs ---

Duplicate: app_042
    _id    full_name annual_income_raw  loan_approved
app_042 Joseph Lopez             69000          False
app_042 Joseph Lopez             69000          False

Duplicate: app_001
    _id        full_name annual_income_raw  loan_approved
app_001 Stephanie Nguyen            102000          False
app_001 Stephanie Nguyen            102000          False


In [130]:
df = df_raw.drop_duplicates(subset='_id', keep='first').copy()

# reset_index(drop=True) renumbers the rows from 0 to N-1.
# After dropping rows, the original row numbers (0-501) have gaps.
# Resetting gives us clean sequential numbers: 0, 1, 2, 3... 499.
# drop=True means: throw away the old index rather than saving it as a column.
df.reset_index(drop=True, inplace=True)
# inplace=True means: modify df directly instead of returning a new DataFrame

print(f'Records BEFORE deduplication: {len(df_raw):,}')
print(f'Records AFTER  deduplication: {len(df):,}')
print(f'Rows removed               : {len(df_raw) - len(df)}')
print(f'\n✓ Duplicate removal complete')




Records BEFORE deduplication: 502
Records AFTER  deduplication: 500
Rows removed               : 2

✓ Duplicate removal complete


## Section 4 — Issue #2: Inconsistent Gender Coding

In [131]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6: Visualise the raw gender values — see the problem with our own eyes
# ═══════════════════════════════════════════════════════════════════════════

# fillna('NULL') temporarily replaces actual Python None values with the
# text string 'NULL' so they show up in the chart.
# Without this, None values would be silently ignored in the count.
raw_gender_counts = df['gender_raw'].fillna('NULL').value_counts()

print('Raw gender values and their counts:')
print(raw_gender_counts.to_string())
# to_string() prints the full Series without truncation


Raw gender values and their counts:
gender_raw
Male      194
Female    193
F          58
M          53
            2


In [132]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7: Standardise all gender values to 'Male', 'Female', or NaN
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Define the mapping (translation table) ──────────────────────────
# A Python dictionary maps each possible raw value to what it should become.
# np.nan is Python's official way to represent "missing / not available".

GENDER_MAP = {
    'Male'   : 'Male',    # already correct, keep as-is
    'M'      : 'Male',    # abbreviation → full word
    'Female' : 'Female',  # already correct, keep as-is
    'F'      : 'Female',  # abbreviation → full word
    ''       : np.nan,    # empty string → unknown (NaN = Not a Number, used for missing)
    None     : np.nan,    # Python None → unknown
}

# ─── Step 2: Apply the mapping to create a new clean column ──────────────────
#
# .map(GENDER_MAP) goes through every value in the 'gender_raw' column,
# looks it up in GENDER_MAP, and returns the corresponding standardised value.
# We store the result in a NEW column called 'gender'.
# df['gender_raw'] is left completely unchanged as our audit trail.

df['gender'] = df['gender_raw'].map(GENDER_MAP)

# ─── Step 3: Verify the result ───────────────────────────────────────────────
print('=== BEFORE vs AFTER Gender Standardisation ===')
print(f'\nBEFORE (raw values):')
print(df['gender_raw'].fillna('NULL').value_counts().to_string())

print(f'\nAFTER (standardised values):')
print(df['gender'].value_counts(dropna=False).to_string())
# dropna=False means: include the NaN count in the output
# By default value_counts() hides NaN — we want to see it

# Count how many were actually changed (had abbreviated form)
changed = (df['gender_raw'].isin(['M', 'F'])).sum()
unknown = df['gender'].isna().sum()

print(f'\nSummary:')
print(f'  Records changed (M→Male or F→Female)  : {changed}')
print(f'  Records set to unknown (NaN)           : {unknown}')
print(f'  Records already correct (no change)   : {len(df) - changed - unknown}')


=== BEFORE vs AFTER Gender Standardisation ===

BEFORE (raw values):
gender_raw
Male      194
Female    193
F          58
M          53
            2

AFTER (standardised values):
gender
Female    251
Male      247
NaN         2

Summary:
  Records changed (M→Male or F→Female)  : 111
  Records set to unknown (NaN)           : 2
  Records already correct (no change)   : 387


## Section 5 — Issue #3: Annual Income Stored as Text Instead of a Number

In [133]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8: Detect records where annual_income is stored as text (string)
# ═══════════════════════════════════════════════════════════════════════════

# ─── Check what data types are actually in the column ─────────────────────────
# .apply(type) applies Python's built-in type() function to every value.
# value_counts() then counts how many of each type there are.

type_breakdown = df['annual_income_raw'].apply(type).value_counts()
print('Data types found in annual_income_raw column:')
print(type_breakdown.to_string())

# ─── Isolate the rows where income is stored as a string ─────────────────────
# isinstance(x, str) returns True if x is a text string, False otherwise
# .apply() runs this check on every value in the column
# The result is a column of True/False values — called a "boolean mask"

string_income_mask = df['annual_income_raw'].apply(lambda x: isinstance(x, str))
# lambda x: ... means: "for each value x, run this small function"
# It is equivalent to writing:
#   def check_if_string(x):
#       return isinstance(x, str)
#   string_income_mask = df['annual_income_raw'].apply(check_if_string)
# But lambdas are more concise for simple one-line functions

string_income_rows = df[string_income_mask][['_id', 'annual_income_raw', 'loan_approved']]

print(f'\nRecords with string income: {string_income_mask.sum()}')
print(string_income_rows.to_string(index=False))


Data types found in annual_income_raw column:
annual_income_raw
<class 'int'>         486
<class 'str'>           8
<class 'NoneType'>      5
<class 'float'>         1

Records with string income: 8
    _id annual_income_raw  loan_approved
app_088             55000          False
app_135             65000          False
app_446             73000           True
app_389             51000           True
app_026             72000           True
app_312             80000           True
app_180            111000          False
app_224             93000          False


In [134]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9: Fix the type mismatch — convert everything to a proper number
# ═══════════════════════════════════════════════════════════════════════════

# pd.to_numeric() converts values to numbers.
#
# errors='coerce' is the key argument:
#   - If a value IS a valid number (like 85000 or "85000"), convert it normally
#   - If a value CANNOT be converted (like "unknown" or None), replace it with NaN
#     instead of crashing with an error

df['annual_income'] = pd.to_numeric(df['annual_income_raw'], errors='coerce')
# We create a NEW column 'annual_income' with the clean numeric values
# 'annual_income_raw' is kept unchanged as our audit trail

# ─── Verify the fix ───────────────────────────────────────────────────────────
# Check data type of the new column
print(f'Data type of annual_income_raw : {df["annual_income_raw"].dtype}')
# 'object' means "mixed types" — pandas uses this for text/mixed columns
print(f'Data type of annual_income     : {df["annual_income"].dtype}')
# 'float64' means 64-bit floating point numbers — the correct numeric type

# Check remaining NaN values (these are the truly missing ones, not the strings)
remaining_nan = df['annual_income'].isna().sum()
print(f'\nNaN values after transformation: {remaining_nan}')
print(f'  → These are the genuinely missing incomes (the None values)')
print(f'  → We will handle them in the next section')

# Show the income range to confirm the numbers look sensible
valid_incomes = df['annual_income'].dropna()
# .dropna() removes NaN values before we compute statistics
print(f'\nIncome range (valid records):')
print(f'  Minimum : ${valid_incomes.min():,.0f}')
print(f'  Maximum : ${valid_incomes.max():,.0f}')
print(f'  Average : ${valid_incomes.mean():,.0f}')
print(f'  Median  : ${valid_incomes.median():,.0f}')




Data type of annual_income_raw : object
Data type of annual_income     : float64

NaN values after transformation: 5
  → These are the genuinely missing incomes (the None values)
  → We will handle them in the next section

Income range (valid records):
  Minimum : $0
  Maximum : $171,000
  Average : $82,694
  Median  : $81,000


## Section 6 — Issue #4: Missing Income Values (Null)
5 records have `annual_income = null` — the field is completely absent. These are not zero-income applicants (which would be stored as `0`). The data literally does not exist for these people. Thus we decided to flag thme and give them the median income.

In [135]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10: Detect and handle missing income values
# ═══════════════════════════════════════════════════════════════════════════

# ─── Identify which rows have missing (NaN) income ───────────────────────────

null_income_mask = df['annual_income'].isna()

# Show those 5 rows
null_income_rows = df[null_income_mask][['_id', 'annual_income_raw', 'loan_approved', 'gender']]
print(f'Records with missing income: {null_income_mask.sum()}')
print(null_income_rows.to_string(index=False))

# ─── Calculate the median income to use for imputation ───────────────────────
# .median() ignores NaN values automatically — it only computes over valid numbers
# This is why we do not need to .dropna() first
income_median = df['annual_income'].median()
print(f'\nMedian income (computed from {(~null_income_mask).sum()} valid records): ${income_median:,.0f}')
# ~ is the "NOT" operator — (~null_income_mask) means "records that are NOT null"


Records with missing income: 5
    _id annual_income_raw  loan_approved gender
app_436              None           True Female
app_421              None           True   Male
app_479              None          False Female
app_463              None          False Female
app_449              None           True Female

Median income (computed from 495 valid records): $81,000


In [136]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11: Apply median imputation + create audit flag column
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Create the flag column BEFORE filling the NaN values ─────────────
# We do this first because once we fill the NaN, we lose the ability to
# detect which rows were imputed.
#
# null_income_mask is True for the 5 rows that need imputation.
# We store it as a column called 'income_imputed' — so True means "this
# income value was imputed (estimated), not measured from the application".

df['income_imputed'] = null_income_mask

# ─── Step 2: Fill NaN values with the median ─────────────────────────────────
# .fillna(value) replaces every NaN in the column with the given value.
# The median we calculated $81,000 gets placed into those 5 rows.

df['annual_income'] = df['annual_income'].fillna(income_median)

# ─── Verify: no NaN values should remain ─────────────────────────────────────
remaining_nan = df['annual_income'].isna().sum()
imputed_count = df['income_imputed'].sum()
# .sum() on a boolean column counts the True values

print(f'Remaining NaN in annual_income: {remaining_nan}  ← should be 0')
print(f'Rows flagged as imputed        : {imputed_count}  ← should be 5')
print(f'\n✓ The 5 missing incomes have been filled with median: ${income_median:,.0f}')
print(f'✓ Flag column income_imputed added — True for those 5 rows')

# Show the 5 fixed rows to confirm
print(f'\nThe 5 imputed rows after fix:')
print(df[df['income_imputed']][['_id', 'annual_income', 'income_imputed', 'loan_approved']].to_string(index=False))

Remaining NaN in annual_income: 0  ← should be 0
Rows flagged as imputed        : 5  ← should be 5

✓ The 5 missing incomes have been filled with median: $81,000
✓ Flag column income_imputed added — True for those 5 rows

The 5 imputed rows after fix:
    _id  annual_income  income_imputed  loan_approved
app_436        81000.0            True           True
app_421        81000.0            True           True
app_479        81000.0            True          False
app_463        81000.0            True          False
app_449        81000.0            True           True


## Section 7 — Issue #5: Negative Credit History Months


We take the **absolute value** — we simply remove the minus sign. So `-10` becomes `10` and `-3` becomes `3`. Both values (10 months and 3 months) are plausible credit history lengths for the types of applicants in this dataset.

In [137]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12: Detect and fix negative credit history months
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect negative values ───────────────────────────────────────────────────
# df['credit_history_months'] < 0 creates a True/False mask
# True = this row has a negative value (a problem)
# False = this row has a zero or positive value (OK)

neg_credit_mask = df['credit_history_months'] < 0

neg_credit_rows = df[neg_credit_mask][
    ['_id', 'credit_history_months', 'annual_income', 'loan_approved']
]

print(f'Records with negative credit_history_months: {neg_credit_mask.sum()}')
print(neg_credit_rows.to_string(index=False))

# Show the full distribution to understand the context
print(f'\nFull distribution of credit_history_months:')
print(f'  Minimum   : {df["credit_history_months"].min()} months  ← the problematic values')
print(f'  Median    : {df["credit_history_months"].median()} months')
print(f'  Maximum   : {df["credit_history_months"].max()} months')
print(f'  Mean      : {df["credit_history_months"].mean():.1f} months')

Records with negative credit_history_months: 2
    _id  credit_history_months  annual_income  loan_approved
app_043                    -10       131000.0           True
app_156                     -3        25000.0          False

Full distribution of credit_history_months:
  Minimum   : -10 months  ← the problematic values
  Median    : 48.5 months
  Maximum   : 133 months
  Mean      : 50.4 months


In [138]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13: Fix negative values by taking the absolute value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Step 1: Add a flag column before making any changes ─────────────────────
# Same principle as with income: record which rows were modified
# so downstream analysts know this value was corrected
df['credit_history_months_flag'] = neg_credit_mask

# ─── Step 2: Apply abs() — take the absolute value ───────────────────────────
# .abs() is a pandas method that removes the minus sign from all negative numbers
# Positive numbers are unchanged: abs(48) = 48
# Negative numbers become positive: abs(-10) = 10
df['credit_history_months'] = df['credit_history_months'].abs()
# Note: we modify this column in-place because there is no separate _raw version
# needed here — the correction is straightforward (just remove the minus sign)

# ─── Verify ───────────────────────────────────────────────────────────────────
print('Fixed rows:')
print(df[df['credit_history_months_flag']][
    ['_id', 'credit_history_months', 'credit_history_months_flag']
].to_string(index=False))

print(f'\nNew minimum credit_history_months: {df["credit_history_months"].min()} months  ← no longer negative')
print(f'✓ All credit history values are now non-negative')

Fixed rows:
    _id  credit_history_months  credit_history_months_flag
app_043                     10                        True
app_156                      3                        True

New minimum credit_history_months: 0 months  ← no longer negative
✓ All credit history values are now non-negative


## Section 8 — Issue #6: Impossible Debt-to-Income Ratio
The most likely explanation is a **decimal point error** — the value should be `0.185` (18.5%), not `1.85` (185%). A DTI of 18.5% is completely normal and consistent with the rest of the dataset's distribution.

In [139]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14: Detect and fix the impossible DTI value
# ═══════════════════════════════════════════════════════════════════════════

# ─── Detect values outside the valid [0, 1] range ────────────────────────────
# We check for TWO impossible conditions using | (OR):
#   1. Greater than 1.0 (more than 100% of income in debt)
#   2. Less than 0 (negative debt — nonsensical)

bad_dti_mask = (df['debt_to_income'] > 1.0) | (df['debt_to_income'] < 0)

bad_dti_rows = df[bad_dti_mask][['_id', 'debt_to_income', 'annual_income', 'loan_approved']]

print(f'Records with DTI outside valid [0, 1] range: {bad_dti_mask.sum()}')
print(bad_dti_rows.to_string(index=False))

# Show the distribution of valid DTI values for comparison
valid_dti = df.loc[~bad_dti_mask, 'debt_to_income']
# ~ is NOT — so ~bad_dti_mask means "rows that do NOT have bad DTI"
# df.loc[condition, column] selects rows matching the condition, then the column
print(f'\nDTI distribution (valid records only):')
print(f'  Min    : {valid_dti.min():.3f}')
print(f'  Median : {valid_dti.median():.3f}')
print(f'  Max    : {valid_dti.max():.3f}')
print(f'  Mean   : {valid_dti.mean():.3f}')
print(f'\nFor reference: 1.85 / 10 = {1.85/10:.3f} ← consistent with valid range')


Records with DTI outside valid [0, 1] range: 1
    _id  debt_to_income  annual_income  loan_approved
app_402            1.85        88000.0           True

DTI distribution (valid records only):
  Min    : 0.050
  Median : 0.230
  Max    : 0.450
  Mean   : 0.242

For reference: 1.85 / 10 = 0.185 ← consistent with valid range


In [140]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15: Apply the decimal point correction
# ═══════════════════════════════════════════════════════════════════════════

# Add audit flag first
df['dti_flag'] = bad_dti_mask

# Fix: divide only the problematic rows by 10
# df.loc[condition, column] = value  →  set a specific subset of cells to a new value
# We only modify rows where bad_dti_mask is True — all other rows are unchanged
df.loc[bad_dti_mask, 'debt_to_income'] = df.loc[bad_dti_mask, 'debt_to_income'] / 10

# Verify
print(f'Fixed row:')
print(df[df['dti_flag']][['_id', 'debt_to_income', 'dti_flag']].to_string(index=False))
print(f'\nNew DTI range: {df["debt_to_income"].min():.3f} – {df["debt_to_income"].max():.3f}  ✓')
print(f'✓ All DTI values now within valid [0, 1] range')


Fixed row:
    _id  debt_to_income  dti_flag
app_402           0.185      True

New DTI range: 0.050 – 0.450  ✓
✓ All DTI values now within valid [0, 1] range


## Section 9 — Issue #7: Three Different Date Formats


The `date_of_birth` field contains dates written in **three completely different formats**

The `DD/MM/YYYY` and `MM/DD/YYYY` formats look identical for days between 1 and 12. For example

We assume **DD/MM/YYYY** for the slash format based on context (European institution — Nova SBE, Lisbon).

Convert all dates to the single international standard: **ISO 8601 (`YYYY-MM-DD`)**, then derive each applicant's `age` in years.


In [141]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 16: Classify what format each date string is using
# ═══════════════════════════════════════════════════════════════════════════

# re.match() checks if a string matches a pattern
# The patterns use "regular expressions" — a mini-language for describing text patterns:
#   \d    = any single digit (0–9)
#   \d{4} = exactly four consecutive digits
#   \d{2} = exactly two consecutive digits
#   -     = a literal hyphen character
#   /     = a literal slash character
#   ^     = start of string
#   $     = end of string
# So '^\d{4}-\d{2}-\d{2}$' means: "exactly 4 digits, hyphen, 2 digits, hyphen, 2 digits"

def classify_date_format(date_string):
    """
    Looks at a date string and identifies which format it uses.
    Returns a label string describing the format.
    """
    if not date_string:               # covers None and empty string ''
        return 'MISSING'
    d = str(date_string)              # ensure it is definitely a string
    if re.match(r'^\d{4}-\d{2}-\d{2}$', d):   # e.g. "1990-03-15"
        return 'YYYY-MM-DD'
    elif re.match(r'^\d{2}/\d{2}/\d{4}$', d): # e.g. "15/03/1990"
        return 'DD/MM/YYYY'
    elif re.match(r'^\d{4}/\d{2}/\d{2}$', d): # e.g. "1990/03/15"
        return 'YYYY/MM/DD'
    else:
        return f'UNKNOWN: {d}'        # catch-all for anything unexpected

# Apply the classification function to every row
df['dob_format'] = df['date_of_birth_raw'].apply(classify_date_format)
format_counts = df['dob_format'].value_counts()

print('Date format breakdown in the raw data:')
print(format_counts.to_string())
print(f'\nRecords with non-standard or missing format: {(df["dob_format"] != "YYYY-MM-DD").sum()}')
print(f'That is {(df["dob_format"] != "YYYY-MM-DD").mean()*100:.1f}% of all records')


Date format breakdown in the raw data:
dob_format
YYYY-MM-DD    339
DD/MM/YYYY    101
YYYY/MM/DD     56
MISSING         4

Records with non-standard or missing format: 161
That is 32.2% of all records


In [142]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 17: Parse all dates into a single standard format, then derive age
# ═══════════════════════════════════════════════════════════════════════════

def parse_date_of_birth(raw_string):
    """
    Tries to parse a date string using all three known formats.
    Returns a Python datetime object if successful, or pd.NaT if it fails.

    pd.NaT = "Not a Time" — pandas' equivalent of NaN for dates.
    It behaves like a missing value in all date operations.

    Parameters
    ----------
    raw_string : str or None
        The raw date string from the JSON file.

    Returns
    -------
    datetime or pd.NaT
    """
    if not raw_string:       # handles None and empty string
        return pd.NaT

    # We try each known format one by one.
    # The order matters — we try the most common format first.
    #
    # datetime.strptime(string, format_string) attempts to parse the string.
    # Format codes:
    #   %Y = 4-digit year (e.g. 1990)
    #   %m = 2-digit month (e.g. 03 for March)
    #   %d = 2-digit day   (e.g. 15)
    # If the string does not match the format, it raises a ValueError — which
    # we catch with 'except ValueError: continue' and try the next format.

    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%Y/%m/%d'):
        try:
            return datetime.strptime(str(raw_string), fmt)
        except ValueError:
            continue  # 'continue' moves to the next iteration of the loop

    # If ALL three formats fail, return NaT (missing date)
    return pd.NaT

# Apply the parser to every row
df['date_of_birth'] = df['date_of_birth_raw'].apply(parse_date_of_birth)

# ─── Derive age from date of birth ───────────────────────────────────────────
# For each valid date of birth, we calculate:
#   age = (AUDIT_DATE - date_of_birth).days ÷ 365
#
# (AUDIT_DATE - date) computes a "timedelta" — the difference between two dates
# .days gives us that difference in days
# We then divide by 365 to convert to years (using integer division // to round down)
#
# For rows where date_of_birth is NaT (missing), we return np.nan

df['age'] = df['date_of_birth'].apply(
    lambda d: (AUDIT_DATE - d).days // 365 if pd.notna(d) else np.nan
)
# pd.notna(d) returns True if d is a real date, False if it is NaT

# ─── Verify ───────────────────────────────────────────────────────────────────
successfully_parsed = df['date_of_birth'].notna().sum()
failed_to_parse     = df['date_of_birth'].isna().sum()

print(f'Dates successfully parsed : {successfully_parsed}')
print(f'Could not parse (NaT)     : {failed_to_parse}  ← these are the missing ones')
print(f'\nAge statistics:')
print(f'  Youngest applicant : {df["age"].min():.0f} years old')
print(f'  Oldest applicant   : {df["age"].max():.0f} years old')
print(f'  Average age        : {df["age"].mean():.1f} years')
print(f'  Median age         : {df["age"].median():.1f} years')

quality_report['issues']['7_inconsistent_date_formats'] = {
    'description'    : 'date_of_birth uses 3 different formats: YYYY-MM-DD, DD/MM/YYYY, YYYY/MM/DD',
    'format_counts'  : {fmt: int(cnt) for fmt, cnt in format_counts.items()},
    'affected_count' : int((df['dob_format'] != 'YYYY-MM-DD').sum()),
    'action'         : 'Multi-format parser normalises all to ISO 8601 datetime. Age column derived.',
    'dimension'      : 'Consistency',
}
print('\n✓ Quality report updated')


Dates successfully parsed : 470
Could not parse (NaT)     : 30  ← these are the missing ones

Age statistics:
  Youngest applicant : 23 years old
  Oldest applicant   : 67 years old
  Average age        : 40.8 years
  Median age         : 39.0 years

✓ Quality report updated
